# Assignement-3

Q. Fine-tune GPT or GPT-2 for creative story generation

Step 1: Install Dependencies

In [2]:
!pip install transformers datasets torch accelerate

Step 2: Import Libraries

In [3]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments
from datasets import load_dataset
import torch

Step 3: Load Dataset

In [4]:
dataset = load_dataset("roneneldan/TinyStories")

# Split dataset
train_dataset = dataset['train'].select(range(5000))  # small subset for Colab

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Step 4: Load Tokenizer

In [5]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Step 5: Tokenization Function

In [6]:
def tokenize_function(examples):
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy() # Add labels for causal language modeling
    return tokenized_inputs

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Step 6: Load GPT-2 Model

In [7]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Step 7: Training Arguments

In [8]:
training_args = TrainingArguments(
    output_dir="./gpt2-story",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
)

Step 8: Trainer Setup

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

Step 9: Train the Model

In [10]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()  # 🔥 IMPORTANT LINE
    return tokens

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [11]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,2.246512
200,2.087399
300,2.019876
400,2.004330
500,2.009618
600,1.961510
700,1.969190
800,1.946115
900,1.921062
1000,1.925999


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=1.9848501953125, metrics={'train_runtime': 243.5161, 'train_samples_per_second': 20.533, 'train_steps_per_second': 5.133, 'total_flos': 326615040000000.0, 'train_loss': 1.9848501953125, 'epoch': 1.0})

Step 10: Save Model

In [12]:
trainer.save_model('./gpt2-story-finetuned')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step 11: Generate Stories

In [14]:
from transformers import pipeline

generator = pipeline("text-generation", model="./gpt2-story-finetuned", tokenizer=tokenizer)

prompt = "Once upon a time in a magical forest"
output = generator(prompt, max_length=100, num_return_sequences=1)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Once upon a time in a magical forest, there was a little rabbit named Timmy. Timmy loved to play outside and explore the forest. One day, Timmy's friend, a big brown wolf, came to his house. 

"Hello, Timmy," said the wolf. "Can I come with you?" 

Timmy nodded and said no. But the wolf said, "Will you let me play with you?" 

"Yes, sweetie," said the wolf. But Timmy didn't want to let the wolf play with him. He wanted to play with his friend, his friend, and his friend's friend. 

Timmy felt very sad and scared. But the wolf was not angry. He hugged Timmy and said, "You are my friend, sweetie. I will play with you and hug you together." The wolf smiled and hugged Timmy and said, "Thank you, sweetie. I will always be with you and hug you with my big blue eyes." 

Timmy and the wolf were so happy. They hugged and hugged for a long time. Suddenly, the wolf's friend, a big, friendly frog, came to visit. The wolf said to Timmy, "Hi, Timmy! Can you
